# Superstore Sales EDA: 04. Strategic Business Insights & Recommendations
---
**Objective:** Deliver direct, quantifiable answers to the 8 Core Business Questions, run financial sensitivity simulations (capping discounts, fixing Tables), and outline an executive action plan.


In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

def get_path(rel_path):
    for candidate in [Path(rel_path), Path("..") / rel_path, Path("/home/adarsh/Documents/Superstore Sales EDA") / rel_path]:
        if candidate.exists():
            return candidate
    return Path(rel_path)

featured_path = get_path("data/cleaned/superstore_featured.csv")
rfm_path = get_path("data/cleaned/customer_rfm.csv")

df = pd.read_csv(featured_path)
df["Order Date"] = pd.to_datetime(df["Order Date"])
rfm = pd.read_csv(rfm_path)
print(f"Loaded featured dataset ({df.shape}) and customer RFM ({rfm.shape}).")


Loaded featured dataset ((9994, 34)) and customer RFM ((793, 15)).


## Business Question 1: Which categories & sub-categories generate profit, and which lose money?


In [2]:
sub_summary = df.groupby(["Category", "Sub-Category"]).agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Orders=("Row ID", "count")
).reset_index()
sub_summary["Margin%"] = (sub_summary["Profit"] / sub_summary["Sales"]) * 100
sub_summary = sub_summary.sort_values(by="Profit", ascending=False)

print("Champion Sub-Categories (Profit > $30K):")
print(sub_summary[sub_summary["Profit"] > 30000])

print("\nLoss-Making Sub-Categories:")
print(sub_summary[sub_summary["Profit"] < 0])


Champion Sub-Categories (Profit > $30K):
           Category Sub-Category       Sales      Profit  Orders    Margin%
14       Technology      Copiers  149528.030  55617.8249      68  37.195585
16       Technology       Phones  330007.054  44515.7306     889  13.489327
13       Technology  Accessories  167380.318  41936.6357     775  25.054700
10  Office Supplies        Paper   78479.206  34053.5693    1370  43.391837
6   Office Supplies      Binders  203412.733  30221.7633    1523  14.857361

Loss-Making Sub-Categories:
           Category Sub-Category        Sales      Profit  Orders   Margin%
12  Office Supplies     Supplies   46673.5380  -1189.0995     190 -2.547695
0         Furniture    Bookcases  114879.9963  -3472.5560     228 -3.022768
3         Furniture       Tables  206965.5320 -17725.4811     319 -8.564460


## Business Question 2: How do discounts affect profit and margin?


In [3]:
disc_analysis = df.groupby("Discount_Bucket", observed=False).agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Orders=("Row ID", "count"),
    Loss_Orders=("Is_Loss", "sum")
).reset_index()
disc_analysis["Margin%"] = (disc_analysis["Profit"] / disc_analysis["Sales"]) * 100
disc_analysis["Loss_Rate%"] = (disc_analysis["Loss_Orders"] / disc_analysis["Orders"]) * 100
print(disc_analysis)


   Discount_Bucket         Sales       Profit  Orders  Loss_Orders  \
0        0% (None)  1.087908e+06  320987.6032    4798            0   
1      1-20% (Low)  8.465222e+05  100785.4745    3803          523   
2  21-50% (Medium)  2.985414e+05  -58817.0047     537          492   
3      50%+ (High)  6.422874e+04  -76559.0513     856          856   

      Margin%  Loss_Rate%  
0   29.505019    0.000000  
1   11.905827   13.752301  
2  -19.701456   91.620112  
3 -119.197502  100.000000  


## Business Question 3: Which regions and states underperform?


In [4]:
reg_analysis = df.groupby("Region").agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum")
).reset_index()
reg_analysis["Margin%"] = (reg_analysis["Profit"] / reg_analysis["Sales"]) * 100
print("Regional Overview:")
print(reg_analysis.sort_values(by="Profit", ascending=False))

loss_states = df.groupby("State").agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Avg_Discount=("Discount", "mean")
).reset_index()
loss_states["Margin%"] = (loss_states["Profit"] / loss_states["Sales"]) * 100
loss_states = loss_states[loss_states["Profit"] < 0].sort_values(by="Profit")
print(f"\nTotal Loss-Making States: {len(loss_states)} states totaling ${loss_states['Profit'].sum():,.2f} in losses:")
print(loss_states)


Regional Overview:
    Region        Sales       Profit    Margin%
3     West  725457.8245  108418.4489  14.944831
1     East  678781.2400   91522.7800  13.483399
2    South  391721.9050   46749.4303  11.934342
0  Central  501239.8908   39706.3625   7.921629

Total Loss-Making States: 10 states totaling $-98,246.74 in losses:
             State        Sales      Profit  Avg_Discount    Margin%
41           Texas  170188.0458 -25729.3563      0.370193 -15.118192
33            Ohio   78258.1360 -16971.3766      0.324947 -21.686405
36    Pennsylvania  116511.9140 -15559.9603      0.328620 -13.354823
11        Illinois   80166.1010 -12607.8870      0.390041 -15.727205
31  North Carolina   55603.1640  -7490.9122      0.283534 -13.472097
4         Colorado   32108.1180  -6527.8579      0.316484 -20.330864
40       Tennessee   30661.8730  -5341.6936      0.291257 -17.421289
1          Arizona   35282.0010  -3427.9246      0.303571  -9.715789
8          Florida   89473.7080  -3399.3017      0.

## Business Question 4: Who are the most valuable customers and segments?


In [5]:
rfm_overview = rfm.groupby("RFM_Segment").agg(
    Customer_Count=("Customer ID", "count"),
    Total_Sales=("Monetary_Sales", "sum"),
    Total_Profit=("Monetary_Profit", "sum")
).reset_index()
rfm_overview["Sales_Share%"] = (rfm_overview["Total_Sales"] / rfm_overview["Total_Sales"].sum()) * 100
rfm_overview["Profit_Share%"] = (rfm_overview["Total_Profit"] / rfm_overview["Total_Profit"].sum()) * 100
rfm_overview["Margin%"] = (rfm_overview["Total_Profit"] / rfm_overview["Total_Sales"]) * 100
print(rfm_overview.sort_values(by="Total_Profit", ascending=False))


                 RFM_Segment  Customer_Count   Total_Sales  Total_Profit  \
3            Loyal Customers             260  1.018149e+06   134907.8634   
1                  Champions             106  5.604988e+05    67795.9212   
4        Potential Loyalists             150  2.943975e+05    35201.1738   
0  At Risk / Needs Attention              98  2.437868e+05    34326.9225   
6     Recent / New Customers              39  5.540808e+04    12188.2048   
2         Hibernating / Lost             117  9.293205e+04     3656.3325   
5        Promising Customers              23  3.202862e+04    -1679.3965   

   Sales_Share%  Profit_Share%    Margin%  
3     44.321286      47.105191  13.250307  
1     24.399208      23.672006  12.095640  
4     12.815487      12.291040  11.957023  
0     10.612343      11.985782  14.080712  
6      2.411983       4.255702  21.997160  
2      4.045447       1.276666   3.934415  
5      1.394246      -0.586388  -5.243424  


## Business Question 5: Monthly, quarterly, and yearly seasonality trends


In [6]:
quarter_analysis = df.groupby(["Order_Year", "Order_Quarter"]).agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum")
).reset_index()
quarter_analysis["Margin%"] = (quarter_analysis["Profit"] / quarter_analysis["Sales"]) * 100
print(quarter_analysis.tail(8))


    Order_Year Order_Quarter        Sales      Profit    Margin%
8         2016            Q1   93237.1810  11441.3708  12.271253
9         2016            Q2  136082.3010  16390.3394  12.044431
10        2016            Q3  143787.3622  15823.6048  11.004865
11        2016            Q4  236098.7538  38139.8593  16.154198
12        2017            Q1  123144.8602  23506.2026  19.088253
13        2017            Q2  133764.3720  15499.2085  11.586948
14        2017            Q3  196251.9560  26985.1325  13.750249
15        2017            Q4  280054.0670  27448.7260   9.801224


## Business Question 6: Shipping mode impact on lead time & profit


In [7]:
ship_summary = df.groupby("Ship Mode").agg(
    Lead_Time_Days=("Shipping_Days", "mean"),
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum")
).assign(Margin_Pct=lambda x: (x["Profit"] / x["Sales"]) * 100)
print(ship_summary)


                Lead_Time_Days         Sales       Profit  Margin_Pct
Ship Mode                                                            
First Class           2.182705  3.514284e+05   48969.8399   13.934513
Same Day              0.044199  1.283631e+05   15891.7589   12.380315
Second Class          3.238046  4.591936e+05   57446.6354   12.510331
Standard Class        5.006535  1.358216e+06  164088.7875   12.081202


## Business Question 7: Chronic loss-making products
Identify individual SKUs with the largest aggregate net losses.


In [8]:
product_losses = df.groupby(["Product ID", "Product Name", "Category", "Sub-Category"]).agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Orders=("Row ID", "count"),
    Avg_Discount=("Discount", "mean")
).reset_index()
product_losses = product_losses.sort_values(by="Profit", ascending=True)

print("Top 10 Chronic Loss-Making Products:")
print(product_losses.head(10))


Top 10 Chronic Loss-Making Products:
           Product ID                                       Product Name  \
1645  TEC-MA-10000418          Cubify CubeX 3D Printer Double Head Print   
1650  TEC-MA-10000822          Lexmark MX611dhe Monochrome Laser Printer   
1696  TEC-MA-10004125          Cubify CubeX 3D Printer Triple Head Print   
326   FUR-TA-10000198  Chromcraft Bull-Nose Wood Oval Conference Tabl...   
344   FUR-TA-10001889  Bush Advantage Collection Racetrack Conference...   
858   OFF-BI-10004995          GBC DocuBind P400 Electric Binding System   
1669  TEC-MA-10002412  Cisco TelePresence System EX90 Videoconferenci...   
1465  OFF-SU-10002881  Martin Yale Chadless Opener Electric Letter Op...   
346   FUR-TA-10001950                       Balt Solid Wood Round Tables   
375   FUR-TA-10004289  BoxOffice By Design Rectangular and Half-Moon ...   

             Category Sub-Category      Sales     Profit  Orders  Avg_Discount  
1645       Technology     Machines  11099.963

## Business Question 8 & Scenario Simulations: How to Increase Profit
What is the quantified impact of eliminating discounts > 20%?


In [9]:
# Current Baseline
baseline_sales = df["Sales"].sum()
baseline_profit = df["Profit"].sum()
baseline_margin = (baseline_profit / baseline_sales) * 100

# Simulation 1: Eliminate transactions with Discount > 20%
df_capped = df[df["Discount"] <= 0.20]
sim1_sales = df_capped["Sales"].sum()
sim1_profit = df_capped["Profit"].sum()
sim1_margin = (sim1_profit / sim1_sales) * 100

# Simulation 2: Eliminate unprofitable Tables sub-category
df_no_loss_tables = df[df["Sub-Category"] != "Tables"]
sim2_sales = df_no_loss_tables["Sales"].sum()
sim2_profit = df_no_loss_tables["Profit"].sum()
sim2_margin = (sim2_profit / sim2_sales) * 100

sim_summary = pd.DataFrame([
    {"Scenario": "Baseline (Current)", "Sales": baseline_sales, "Profit": baseline_profit, "Margin%": baseline_margin},
    {"Scenario": "Sim 1: Cap Discounts at 20%", "Sales": sim1_sales, "Profit": sim1_profit, "Margin%": sim1_margin},
    {"Scenario": "Sim 2: Exclude Tables Sub-Category", "Sales": sim2_sales, "Profit": sim2_profit, "Margin%": sim2_margin},
    {"Scenario": "Sim 3: Cap Discounts at 20% & Optimize Tables", 
     "Sales": df[(df["Discount"] <= 0.20) | (df["Sub-Category"] != "Tables")]["Sales"].sum(),
     "Profit": df_capped[df_capped["Sub-Category"] != "Tables"]["Profit"].sum(),
     "Margin%": (df_capped[df_capped["Sub-Category"] != "Tables"]["Profit"].sum() / df_capped[df_capped["Sub-Category"] != "Tables"]["Sales"].sum()) * 100}
])

sim_summary["Profit_Lift ($)"] = sim_summary["Profit"] - baseline_profit
sim_summary["Profit_Growth (%)"] = (sim_summary["Profit_Lift ($)"] / baseline_profit) * 100
print(sim_summary)


                                        Scenario         Sales       Profit  \
0                             Baseline (Current)  2.297201e+06  286397.0217   
1                    Sim 1: Cap Discounts at 20%  1.934431e+06  421773.0777   
2             Sim 2: Exclude Tables Sub-Category  2.090235e+06  304122.5028   
3  Sim 3: Cap Discounts at 20% & Optimize Tables  2.207244e+06  408800.3360   

     Margin%  Profit_Lift ($)  Profit_Growth (%)  
0  12.467217           0.0000           0.000000  
1  21.803473      135376.0560          47.268668  
2  14.549678       17725.4811           6.189129  
3  22.493422      122403.3143          42.739032  
